# PillSight — Model 3: Vision Transformer (ViT-B/16)

**ADSP 31018 | University of Chicago**

This notebook runs the full ViT pipeline:
1. GPU check
2. Clone repo + install deps
3. Download ePillID dataset
4. Run `data_exploration.py` (shared with Model 1)
5. Train ViT-B/16
6. Evaluate on test set
7. Generate attention maps

> **Runtime → Change runtime type → T4 GPU** before running!

## 0. Confirm GPU

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('⚠️  No GPU detected — go to Runtime → Change runtime type → T4 GPU')

## 1. Clone repo & install dependencies

In [ ]:
import os

REPO_URL  = 'https://github.com/Devanshu1503/ML2_Class_Project'
REPO_NAME = 'ML2_Class_Project'

if not os.path.exists(REPO_NAME):
    !git clone {REPO_URL}
else:
    print('Repo already cloned — pulling latest')
    !cd {REPO_NAME} && git pull

# Set working dir so all relative paths resolve correctly
os.chdir(f'/content/{REPO_NAME}')
print('CWD:', os.getcwd())

In [ ]:
!pip install -q -r model3_vit/requirements.txt
print('✓ Dependencies installed')

## 2. Download the ePillID dataset

The dataset is ~153 MB from the NIH. We download it once and cache it.

In [ ]:
import zipfile, pathlib

DATA_URL = 'https://github.com/usuyama/ePillID-benchmark/releases/download/ePillID_data_v1.0/ePillID_data.zip'
ZIP_PATH = '/content/ePillID_data.zip'
DATA_DIR = '/content/ML2_Class_Project/data'

if not pathlib.Path(DATA_DIR).exists():
    print('Downloading ePillID dataset (~153 MB)...')
    !wget -q --show-progress -O {ZIP_PATH} {DATA_URL}
    print('Unzipping...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall('/content/ML2_Class_Project/')
    print('✓ Dataset ready at', DATA_DIR)
else:
    print('✓ Dataset already present')

# Quick sanity check
!ls data/

## 3. Set COLAB_ROOT so all scripts find the data folder

Our `config.py` reads this env var to locate the project root.

In [ ]:
import os
os.environ['COLAB_ROOT'] = '/content/ML2_Class_Project'
print('COLAB_ROOT set to:', os.environ['COLAB_ROOT'])

## 4. Filter dataset & create class map

Reuses the shared `data_exploration.py` from Model 1 to write
`filtered_metadata.csv` and `class_map.json` into the `data/` folder.

**Skip this step if you already ran Model 1** — the files will already be there.

In [ ]:
import pathlib

filtered_csv = pathlib.Path('data/filtered_metadata.csv')
if filtered_csv.exists():
    print('✓ filtered_metadata.csv already exists — skipping data_exploration.py')
else:
    print('Running data_exploration.py from model1_cnn...')
    !cd model1_cnn && python data_exploration.py
    print('✓ Done')

## 5. Train ViT-B/16

In [ ]:
# Optional: tweak hyperparameters without editing config.py
# They're read at import time, so set them before the import.

# Example overrides (uncomment to use):
# import os
# os.environ['VIT_EPOCHS']   = '10'   # faster smoke-test
# os.environ['VIT_BATCH']    = '16'   # if you hit OOM on a small GPU

os.chdir('/content/ML2_Class_Project/model3_vit')
print('Running train.py...')
!python train.py

## 6. Evaluate on test set

In [ ]:
os.chdir('/content/ML2_Class_Project/model3_vit')
!python evaluate.py

## 7. Attention map visualizations

In [ ]:
os.chdir('/content/ML2_Class_Project/model3_vit')
!python attention_map.py

## 8. Display results inline

In [ ]:
import json, glob
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

BASE = '/content/ML2_Class_Project/model3_vit'

# Learning curves
lc_path = f'{BASE}/outputs/learning_curves.png'
if os.path.exists(lc_path):
    plt.figure(figsize=(10, 4))
    plt.imshow(mpimg.imread(lc_path))
    plt.axis('off'); plt.title('Learning Curves'); plt.show()

# Metrics
m_path = f'{BASE}/outputs/metrics.json'
if os.path.exists(m_path):
    metrics = json.load(open(m_path))
    print('\n── Test metrics ──────────────────')
    for k, v in metrics.items():
        print(f'  {k:20s}: {v:.4f}')

# Attention maps
attn_imgs = sorted(glob.glob(f'{BASE}/outputs/attention_maps/*.png'))[:3]
if attn_imgs:
    fig, axes = plt.subplots(1, len(attn_imgs), figsize=(14, 5))
    if len(attn_imgs) == 1: axes = [axes]
    for ax, p in zip(axes, attn_imgs):
        ax.imshow(mpimg.imread(p))
        ax.axis('off')
        ax.set_title(os.path.basename(p), fontsize=7)
    plt.suptitle('Attention Maps (Attention Rollout)', fontsize=10)
    plt.tight_layout(); plt.show()

## 9. Save outputs to Google Drive (optional)

Colab VMs reset on disconnect. Mount Drive to persist your checkpoint.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
#
# import shutil
# DRIVE_DEST = '/content/drive/MyDrive/ML2_PillSight/model3_vit'
# shutil.copytree(
#     '/content/ML2_Class_Project/model3_vit/checkpoints',
#     f'{DRIVE_DEST}/checkpoints',
#     dirs_exist_ok=True
# )
# shutil.copytree(
#     '/content/ML2_Class_Project/model3_vit/outputs',
#     f'{DRIVE_DEST}/outputs',
#     dirs_exist_ok=True
# )
# print('✓ Saved to Google Drive')
print('Uncomment the block above to save to Drive.')